In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"E:\Hybrid_IDS_Project\dataset\CICIDS2017_Cleaned.csv"
)

df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)

Dataset shape: (2520798, 79)


In [2]:
X = df.drop("Label", axis=1)

y = np.where(df["Label"] == "BENIGN", 0, 1)

print("X:", X.shape)
print("y:", y.shape)

print(pd.Series(y).value_counts())

X: (2520798, 78)
y: (2520798,)
0    2095057
1     425741
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (2016638, 78)
Testing : (504160, 78)


In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training scaled:", X_train_scaled.shape)
print("Testing scaled :", X_test_scaled.shape)

Training scaled: (2016638, 78)
Testing scaled : (504160, 78)


In [5]:
X_train_scaled = X_train_scaled.astype("float32")
X_test_scaled = X_test_scaled.astype("float32")

y_train = np.asarray(y_train).astype("float32")
y_test = np.asarray(y_test).astype("float32")

In [6]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.21.0
GPU: []


Build the MLP Model

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

model = Sequential([
    Input(shape=(78,)),          # 78 input features

    Dense(128, activation='relu'),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        10,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,481 (80.00 KB)

 Trainable params: 20,481 (80.00 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [9]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight_dict = {
    0: class_weights[0],
    1: class_weights[1]
}

print(class_weight_dict)

{0: np.float64(0.6016061621257186), 1: np.float64(2.9604806910300563)}


In [10]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    r"E:\Hybrid_IDS_Project\dataset\models\best_mlp.keras",
    monitor="val_accuracy",
    save_best_only=True
)

Train the MLP

In [11]:
history = model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=1024,
    class_weight=class_weight_dict,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

Epoch 1/20
1576/1576 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.9532 - loss: 0.1043 - val_accuracy: 0.9590 - val_loss: 0.0809
Epoch 2/20
1576/1576 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9678 - loss: 0.0657 - val_accuracy: 0.9723 - val_loss: 0.0545
Epoch 3/20
1576/1576 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9706 - loss: 0.0587 - val_accuracy: 0.9755 - val_loss: 0.0548
Epoch 4/20
1576/1576 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9721 - loss: 0.0554 - val_accuracy: 0.9766 - val_loss: 0.0486
Epoch 5/20
1576/1576 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.9726 - loss: 0.0538 - val_accuracy: 0.9700 - val_loss: 0.0537
Epoch 6/20
1576/1576 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.9725 - loss: 0.0531 - val_accuracy: 0.9805 - val_loss: 0.0453
Epoch 7/20
1576/1576 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.9725 - loss: 0.0525 - val_accuracy: 0.9813 - val_loss: 0.0452
Epoch 8/20
1576/1576 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.9726 - loss: 0.05

Evaluate the MLP

In [12]:
y_pred_prob = model.predict(X_test_scaled)

y_pred = (y_pred_prob > 0.5).astype(int)

15755/15755 ━━━━━━━━━━━━━━━━━━━━ 11s 702us/step


In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred,
    target_names=["BENIGN","ATTACK"]
))

Accuracy : 0.9783540939384322
Precision: 0.8974249156807109
Recall   : 0.9843449053412882
F1-score : 0.9388774692931114

Classification Report
              precision    recall  f1-score   support

      BENIGN       1.00      0.98      0.99    419012
      ATTACK       0.90      0.98      0.94     85148

    accuracy                           0.98    504160
   macro avg       0.95      0.98      0.96    504160
weighted avg       0.98      0.98      0.98    504160



In [14]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

fpr = fp / (fp + tn)

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

print("False Positive Rate:", fpr)

TN: 409432
FP: 9580
FN: 1333
TP: 83815
False Positive Rate: 0.02286330701746012
